In [ ]:
from pathlib import Path
import torch

ROOT = Path.cwd()

CONFIG_PATH = ROOT / "config.py"
CHECKPOINT_PATH = ROOT / "outputs" / "epoch_100.pth"

DATASET_ROOT = (
    ROOT / "../../dataset/Circuit testing 2 -test.v1i.coco"
).resolve()

print("Config     :", CONFIG_PATH)
print("Checkpoint :", CHECKPOINT_PATH)
print("Dataset    :", DATASET_ROOT)
print("GPU        :", torch.cuda.get_device_name(0))

assert CONFIG_PATH.exists(), "Config 파일 없음"
assert CHECKPOINT_PATH.exists(), "Checkpoint 파일 없음"
assert (DATASET_ROOT / "test").exists(), "Test dataset 없음"

print("\n준비 완료!")

In [ ]:
from mmdet.apis import init_detector

model = init_detector(
    str(CONFIG_PATH),
    str(CHECKPOINT_PATH),
    device="cuda:0"
)

print("RTMDet 모델 로드 완료!")
print("Classes:", model.dataset_meta["classes"])

In [ ]:
from mmengine.config import Config
from mmengine.runner import Runner

# 저장된 학습 config 불러오기
test_cfg = Config.fromfile(str(CONFIG_PATH))

# 학습된 모델 지정
test_cfg.load_from = str(CHECKPOINT_PATH)

# test에서는 학습 안 하므로 work_dir만 별도로
test_cfg.work_dir = str(ROOT / "test_outputs")

# Runner 생성
runner = Runner.from_cfg(test_cfg)

# Test 평가
metrics = runner.test()

print("\n=== Test Result ===")
print(metrics)

In [ ]:
from mmdet.apis import inference_detector
import matplotlib.pyplot as plt
import cv2

# test 이미지 목록
test_images = sorted(
    list((DATASET_ROOT / "test").glob("*.jpg")) +
    list((DATASET_ROOT / "test").glob("*.png"))
)

print("Test images:", len(test_images))

# 첫 번째 이미지
img_path = test_images[0]

# inference
result = inference_detector(model, str(img_path))

print("Image:", img_path.name)
print("Detected objects:", len(result.pred_instances))

In [ ]:
from mmdet.apis import inference_detector
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

classes = model.dataset_meta["classes"]
CONF_THRESHOLD = 0.5

# ============================================================
# 클래스별 색상
# ============================================================
CLASS_COLORS = {
    "black_wire":     "black",
    "capacitor":      "magenta",
    "ic_chip":        "green",
    "led":            "gold",
    "red_wire":       "red",
    "resistance330":  "darkorange",
    "resistance470":  "blue",
    "white_wire":     "deepskyblue",
}


for img_path in test_images:

    # ========================================================
    # Inference
    # ========================================================
    result = inference_detector(model, str(img_path))

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    pred = result.pred_instances.cpu()

    bboxes = pred.bboxes.numpy()
    scores = pred.scores.numpy()
    labels = pred.labels.numpy()

    # threshold 적용
    detections = [
        (bbox, score, label)
        for bbox, score, label in zip(bboxes, scores, labels)
        if score >= CONF_THRESHOLD
    ]

    # ========================================================
    # Figure
    # ========================================================
    fig, ax = plt.subplots(figsize=(17, 10))

    ax.imshow(img)

    # 오른쪽 공간 확보
    fig.subplots_adjust(right=0.72)

    # ========================================================
    # Bounding Box
    # ========================================================
    for idx, (bbox, score, label) in enumerate(detections, start=1):

        x1, y1, x2, y2 = bbox

        class_name = classes[label]
        color = CLASS_COLORS[class_name]

        # -------------------------
        # 클래스 색상의 Bounding Box
        # -------------------------
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor=color,
            facecolor="none"
        )

        ax.add_patch(rect)

        # -------------------------
        # 사진에는 번호만 표시
        # -------------------------
        ax.text(
            x1,
            max(y1 - 4, 5),
            str(idx),
            fontsize=9,
            fontweight="bold",
            color=color,
            bbox=dict(
                facecolor="white",
                edgecolor=color,
                alpha=0.85,
                pad=1
            )
        )

    # ========================================================
    # 오른쪽 Detection Results
    # ========================================================
    fig.text(
        0.74,
        0.92,
        "Detection Results",
        fontsize=14,
        fontweight="bold",
        verticalalignment="top"
    )

    # 한 줄씩 그려야 클래스별 색상 적용 가능
    y = 0.87

    for idx, (bbox, score, label) in enumerate(detections, start=1):

        class_name = classes[label]
        color = CLASS_COLORS[class_name]

        fig.text(
            0.74,
            y,
            f"{idx:02d}. {class_name:<15} {score:.2f}",
            fontsize=11,
            family="monospace",
            color=color,
            fontweight="bold"
        )

        y -= 0.035

    # ========================================================
    # 출력
    # ========================================================
    ax.set_title(img_path.name)
    ax.axis("off")

    plt.show()